# 01 · From a village microgrid to a QUBO

**Track:** Climate, Energy & Agriculture · QFF 2026

Every hour, the operator of a small off-grid microgrid decides which resources to switch on: rooftop solar,
a wind turbine, battery banks, a diesel generator. Too little and households lose power. Too much and clean
energy is curtailed or diesel is burnt for nothing.

This notebook builds one 6-node instance, writes the decision as a QUBO, maps it to an Ising Hamiltonian
and solves it exactly so we know the right answer before any quantum circuit runs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

## 1. A synthetic microgrid

All numbers are synthetic and seeded (see `src/microgrid/instance.py` for the assumed ranges).

In [ ]:
from microgrid import make_microgrid

grid = make_microgrid(n=6, density=0.3, seed=0)
print(f"{'node':>4} {'resource':>8} {'capacity kW':>12} {'cost $/kWh':>11} {'demand kW':>10}")
for j in range(grid.n):
    print(f"{j:>4} {grid.kinds[j]:>8} {grid.capacity_kw[j]:>12.1f} {grid.cost_per_kwh[j]:>11.3f} {grid.demand_kw[j]:>10.1f}")
print("feeder links:", grid.edges)

In [ ]:
colors = {"solar": "#E8A400", "wind": "#2F8FCE", "battery": "#7A5CF0", "diesel": "#5E5E5E"}
fig, ax = plt.subplots(figsize=(5, 4))
for i, j in grid.edges:
    ax.plot(*zip(grid.positions[i], grid.positions[j]), color="#B9C2CC")
for j in range(grid.n):
    ax.scatter(*grid.positions[j], s=8 * grid.capacity_kw[j], color=colors[grid.kinds[j]], zorder=3)
    ax.annotate(f"{j}: {grid.kinds[j]}", grid.positions[j], xytext=(6, 6), textcoords="offset points")
ax.set_axis_off()
ax.set_title("Feeder graph (marker size = capacity)")
plt.show()

## 2. The objective

$x_j = 1$ switches node $j$'s resource on. A dispatched source shares its output with itself and its direct
feeder neighbours in proportion to their demand; $A_{ij}$ is what node $i$ receives from node $j$.

$$E(x) = \underbrace{\sum_j c_j p_j x_j}_{\text{generation cost}} + \lambda \sum_i \Big(\underbrace{\sum_j A_{ij} x_j}_{\text{supply at } i} - (1+r)\,d_i\Big)^2$$

* $r = 15\%$ operating reserve. The quadratic penalty is symmetric, so the reserve is what tilts the optimum towards keeping the lights on.
* $\lambda$ is set so a mismatch of half the mean node demand costs the value of lost load at the margin (VOLL = 2 USD/kWh, an assumption).

Expanding the square gives a QUBO: $E(x) = \text{const} + h^\top x + x^\top J x$.

In [ ]:
from microgrid import build_qubo
from microgrid.qubo import all_bitstrings, objective_direct

qubo = build_qubo(grid)
bits = all_bitstrings(grid.n)
assert np.allclose(qubo.energies(bits), [objective_direct(grid, x) for x in bits])
print("linear terms h:", np.round(qubo.linear, 2))
print("non-zero couplings:", qubo.num_couplings(), "of", grid.n * (grid.n - 1) // 2)

## 3. Ising Hamiltonian

With $x_j = (1 - Z_j)/2$ the QUBO becomes a sum of $Z$ and $ZZ$ terms plus a constant offset. The test below checks every one of the $2^6$ basis states.

In [ ]:
from microgrid import to_ising

H, offset = to_ising(qubo)
print(H)
print("offset:", round(offset, 3))
diagonal = np.real(H.to_matrix(sparse=True).diagonal()) + offset
assert np.allclose(diagonal, qubo.energies(bits))
print("Ising diagonal matches the QUBO on all", len(bits), "bitstrings")

## 4. Ground truth and classical baselines

In [ ]:
from microgrid import brute_force, greedy_merit_order, simulated_annealing, dispatch_report
from microgrid.qubo import array_to_bitstring

exact, landscape = brute_force(qubo)
for sol in [exact, greedy_merit_order(grid, qubo), simulated_annealing(qubo, seed=0)]:
    rep = dispatch_report(grid, sol.x)
    print(f"{sol.method:22s} x={array_to_bitstring(sol.x)}  objective={sol.energy:8.2f}  "
          f"demand met={rep['pct_demand_met']:5.1f}%  surplus={rep['surplus_kw']:5.1f} kW  "
          f"diesel={rep['diesel_kw']:4.0f} kW  cost={rep['cost_usd_per_h']:5.2f} $/h")

At 6 to 12 nodes the exact optimum is cheap to compute, which is the point: it lets us measure QAOA honestly. Next notebook: how close QAOA gets, with and without noise.